# Notebook 5: Governance, Compliance y Monitoring

## Objetivos
- Implementar frameworks de compliance regulatorio
- Desarrollar sistemas de audit logging comprehensivos
- Crear mecanismos de monitoring de seguridad y ética
- Establecer estructuras de governance para agentes de IA
- Aplicar prácticas de monitoring continuo al agente existente

## Configuración del Agente

In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

## 1. Marcos Regulatorios y Compliance

Implementamos un sistema de compliance con regulaciones clave:

In [ ]:
import time
import hashlib
from datetime import datetime, timedelta

class ComplianceFramework:
    """
    Framework de compliance con regulaciones internacionales.
    """
    
    def __init__(self):
        # Regulaciones soportadas
        self.regulations = {
            "GDPR": {
                "name": "General Data Protection Regulation",
                "region": "UE",
                "key_requirements": [
                    "Consentimiento explícito",
                    "Derecho al olvido",
                    "Portabilidad de datos",
                    "Notificación de brechas",
                    "Derecho a explicación (Art. 22)"
                ]
            },
            "CCPA": {
                "name": "California Consumer Privacy Act",
                "region": "California, USA",
                "key_requirements": [
                    "Derecho a saber",
                    "Derecho a eliminar",
                    "Derecho a opt-out",
                    "Derecho a no discriminación",
                    "Portabilidad de datos"
                ]
            },
            "EU_AI_Act": {
                "name": "European Union AI Act",
                "region": "UE",
                "key_requirements": [
                    "Clasificación de riesgo",
                    "Requisitos de transparencia",
                    "Gobernanza de datos",
                    "Supervisión humana",
                    "Documentación técnica"
                ]
            },
            "SOC_2": {
                "name": "Service Organization Control 2",
                "region": "Global",
                "key_requirements": [
                    "Seguridad",
                    "Disponibilidad",
                    "Integridad",
                    "Confidencialidad",
                    "Privacidad"
                ]
            }
        }
        
        self.compliance_records = {}
        self.audit_trail = []
    
    def check_compliance(self, regulation):
        """
        Verifica el estado de compliance con una regulación específica.
        
        Args:
            regulation: Código de la regulación
        
        Returns:
            dict con estado de compliance
        """
        if regulation not in self.regulations:
            return {
                "status": "not_applicable",
                "message": f"Regulación '{regulation}' no reconocida"
            }
        
        reg_info = self.regulations[regulation]
        compliance_status = self.compliance_records.get(regulation, {})
        
        return {
            "status": "compliant" if compliance_status.get("compliant", False) else "pending",
            "regulation": reg_info["name"],
            "region": reg_info["region"],
            "requirements": reg_info["key_requirements"],
            "last_audit": compliance_status.get("last_audit", None),
            "next_audit": compliance_status.get("next_audit", None)
        }
    
    def update_compliance_status(self, regulation, compliant, notes=""):
        """
        Actualiza el estado de compliance.
        
        Args:
            regulation: Código de la regulación
            compliant: Estado de compliance
            notes: Notas adicionales
        """
        self.compliance_records[regulation] = {
            "compliant": compliant,
            "last_audit": datetime.now().isoformat(),
            "next_audit": (datetime.now() + timedelta(days=365)).isoformat(),
            "notes": notes
        }
        
        self._audit_log(f"Compliance status updated for {regulation}: {compliant}")
    
    def _audit_log(self, message):
        """
        Registra en el audit trail.
        """
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "message": message,
            "hash": hashlib.sha256(message.encode()).hexdigest()[:16]
        }
        self.audit_trail.append(log_entry)
    
    def get_audit_trail(self, limit=None):
        """
        Obtiene el audit trail.
        
        Args:
            limit: Límite de entradas a retornar
        
        Returns:
            list de entradas de audit trail
        """
        if limit:
            return self.audit_trail[-limit:]
        return self.audit_trail

# Crear instancia del ComplianceFramework
compliance_framework = ComplianceFramework()

# Actualizar algunos estados de compliance
compliance_framework.update_compliance_status("GDPR", True, "Implementados todos los requisitos de consentimiento y portabilidad")
compliance_framework.update_compliance_status("SOC_2", True, "Controles de seguridad y disponibilidad implementados")

print("✅ Framework de Compliance configurado.")
print(f"Regulaciones soportadas: {len(compliance_framework.regulations)}")

## 2. Sistema de Audit Logging Comprehensivo

Implementamos un sistema completo de audit logging:

In [ ]:
class ComprehensiveAuditLogger:
    """
    Sistema comprehensivo de audit logging para agentes de IA.
    """
    
    def __init__(self):
        self.audit_log = []
        self.log_categories = {
            "security": "Eventos de seguridad",
            "ethical": "Decisiones éticas",
            "compliance": "Eventos de compliance",
            "performance": "Métricas de rendimiento",
            "user_activity": "Actividad de usuarios"
        }
    
    def log_event(self, category, event_type, details, user_id=None, severity="info"):
        """
        Registra un evento en el audit log.
        
        Args:
            category: Categoría del evento
            event_type: Tipo de evento
            details: Detalles del evento
            user_id: ID del usuario (opcional)
            severity: Severidad del evento
        """
        if category not in self.log_categories:
            raise ValueError(f"Categoría '{category}' no reconocida")
        
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "category": category,
            "event_type": event_type,
            "details": details,
            "user_id": user_id,
            "severity": severity,
            "entry_id": hashlib.sha256(
                f"{datetime.now().isoformat()}{category}{event_type}".encode()
            ).hexdigest()[:16]
        }
        
        self.audit_log.append(log_entry)
    
    def log_security_event(self, event_type, details, user_id=None, severity="warning"):
        """
        Registra un evento de seguridad.
        """
        self.log_event("security", event_type, details, user_id, severity)
    
    def log_ethical_decision(self, event_type, details, user_id=None, severity="info"):
        """
        Registra una decisión ética.
        """
        self.log_event("ethical", event_type, details, user_id, severity)
    
    def log_compliance_event(self, event_type, details, user_id=None, severity="info"):
        """
        Registra un evento de compliance.
        """
        self.log_event("compliance", event_type, details, user_id, severity)
    
    def get_logs(self, category=None, user_id=None, severity=None, limit=None):
        """
        Obtiene logs filtrados.
        
        Args:
            category: Filtrar por categoría
            user_id: Filtrar por usuario
            severity: Filtrar por severidad
            limit: Límite de resultados
        
        Returns:
            list de entradas filtradas
        """
        filtered = self.audit_log
        
        if category:
            filtered = [entry for entry in filtered if entry["category"] == category]
        if user_id:
            filtered = [entry for entry in filtered if entry["user_id"] == user_id]
        if severity:
            filtered = [entry for entry in filtered if entry["severity"] == severity]
        
        if limit:
            filtered = filtered[-limit:]
        
        return filtered
    
    def get_statistics(self):
        """
        Obtiene estadísticas del audit log.
        
        Returns:
            dict con estadísticas
        """
        stats = {
            "total_events": len(self.audit_log),
            "by_category": {},
            "by_severity": {},
            "by_user": {}
        }
        
        for entry in self.audit_log:
            # Por categoría
            category = entry["category"]
            stats["by_category"][category] = stats["by_category"].get(category, 0) + 1
            
            # Por severidad
            severity = entry["severity"]
            stats["by_severity"][severity] = stats["by_severity"].get(severity, 0) + 1
            
            # Por usuario
            user_id = entry["user_id"]
            if user_id:
                stats["by_user"][user_id] = stats["by_user"].get(user_id, 0) + 1
        
        return stats

# Crear instancia del ComprehensiveAuditLogger
audit_logger = ComprehensiveAuditLogger()

print("✅ Sistema Comprehensivo de Audit Logging configurado.")
print("Categorías de logging:")
for category, description in audit_logger.log_categories.items():
    print(f"  • {category}: {description}")

## 3. Sistema de Monitoring de Seguridad

Implementamos un sistema de monitoring de seguridad en tiempo real:

In [ ]:
class SecurityMonitor:
    """
    Sistema de monitoring de seguridad para agentes de IA.
    """
    
    def __init__(self):
        self.threat_indicators = []
        self.behavioral_baselines = {}
        self.alerts = []
        self.security_metrics = {
            "total_requests": 0,
            "blocked_requests": 0,
            "suspicious_activities": 0,
            "security_incidents": 0
        }
    
    def detect_anomalies(self, user_id, request_pattern):
        """
        Detecta comportamiento anómalo de usuarios.
        
        Args:
            user_id: ID del usuario
            request_pattern: Patrón de request
        
        Returns:
            dict con resultado de detección
        """
        baseline = self.behavioral_baselines.get(user_id, self._get_default_baseline())
        anomalies = []
        
        # Frecuencia inusual de requests
        if request_pattern.get('frequency', 0) > baseline.get('avg_frequency', 0) * 3:
            anomalies.append({
                "type": "high_frequency",
                "value": request_pattern['frequency'],
                "baseline": baseline['avg_frequency']
            })
        
        # Complejidad inusual de queries
        if request_pattern.get('complexity', 0) > baseline.get('avg_complexity', 0) * 2:
            anomalies.append({
                "type": "high_complexity",
                "value": request_pattern['complexity'],
                "baseline": baseline['avg_complexity']
            })
        
        result = {
            "has_anomalies": len(anomalies) > 0,
            "anomalies": anomalies,
            "severity": "high" if len(anomalies) > 2 else "medium" if len(anomalies) > 0 else "none"
        }
        
        if result["has_anomalies"]:
            self._create_alert(user_id, "anomaly_detected", result)
        
        return result
    
    def _get_default_baseline(self):
        """
        Obtiene el baseline por defecto.
        """
        return {
            "avg_frequency": 10,
            "avg_complexity": 5,
            "night_activity": 0.05
        }
    
    def update_baseline(self, user_id, request_pattern):
        """
        Actualiza el baseline de comportamiento de un usuario.
        
        Args:
            user_id: ID del usuario
            request_pattern: Patrón de request
        """
        if user_id not in self.behavioral_baselines:
            self.behavioral_baselines[user_id] = self._get_default_baseline()
        
        baseline = self.behavioral_baselines[user_id]
        
        # Actualizar promedios con factor de aprendizaje
        learning_rate = 0.1
        baseline["avg_frequency"] = (
            baseline["avg_frequency"] * (1 - learning_rate) + 
            request_pattern.get('frequency', 0) * learning_rate
        )
        baseline["avg_complexity"] = (
            baseline["avg_complexity"] * (1 - learning_rate) + 
            request_pattern.get('complexity', 0) * learning_rate
        )
    
    def _create_alert(self, user_id, alert_type, details):
        """
        Crea una alerta de seguridad.
        
        Args:
            user_id: ID del usuario
            alert_type: Tipo de alerta
            details: Detalles de la alerta
        """
        alert = {
            "timestamp": datetime.now().isoformat(),
            "user_id": user_id,
            "alert_type": alert_type,
            "details": details,
            "status": "active",
            "alert_id": hashlib.sha256(
                f"{datetime.now().isoformat()}{user_id}{alert_type}".encode()
            ).hexdigest()[:16]
        }
        self.alerts.append(alert)
        self.security_metrics["suspicious_activities"] += 1
    
    def get_active_alerts(self):
        """
        Obtiene alertas activas.
        
        Returns:
            list de alertas activas
        """
        return [alert for alert in self.alerts if alert["status"] == "active"]
    
    def get_security_metrics(self):
        """
        Obtiene métricas de seguridad.
        
        Returns:
            dict con métricas
        """
        return self.security_metrics

# Crear instancia del SecurityMonitor
security_monitor = SecurityMonitor()

print("✅ Sistema de Monitoring de Seguridad configurado.")

## 4. Sistema de Monitoring Ético

Implementamos un sistema de monitoring de comportamiento ético:

In [ ]:
class EthicsMonitor:
    """
    Sistema de monitoring de comportamiento ético.
    """
    
    def __init__(self):
        self.ethics_metrics = {
            "total_decisions": 0,
            "allowed_decisions": 0,
            "blocked_decisions": 0,
            "warning_decisions": 0
        }
        self.bias_indicators = []
        self.demographic_responses = {}
    
    def track_decision(self, decision_type, outcome, user_demographic=None):
        """
        Registra una decisión ética.
        
        Args:
            decision_type: Tipo de decisión
            outcome: Outcome de la decisión
            user_demographic: Demográfico del usuario (opcional)
        """
        self.ethics_metrics["total_decisions"] += 1
        
        if outcome == "allowed":
            self.ethics_metrics["allowed_decisions"] += 1
        elif outcome == "blocked":
            self.ethics_metrics["blocked_decisions"] += 1
        elif outcome == "warning":
            self.ethics_metrics["warning_decisions"] += 1
        
        if user_demographic:
            if user_demographic not in self.demographic_responses:
                self.demographic_responses[user_demographic] = []
            self.demographic_responses[user_demographic].append({
                "decision_type": decision_type,
                "outcome": outcome,
                "timestamp": datetime.now().isoformat()
            })
    
    def get_ethics_metrics(self):
        """
        Obtiene métricas éticas.
        
        Returns:
            dict con métricas
        """
        return self.ethics_metrics

# Crear instancia del EthicsMonitor
ethics_monitor = EthicsMonitor()

print("✅ Sistema de Monitoring Ético configurado.")

## 5. Agente con Governance Completo - Integración Final

Integramos todos los componentes de governance en un wrapper final:

In [ ]:
class GovernedAgentWrapper:
    """
    Wrapper que añade governance completo al agente de IA.
    """
    
    def __init__(self, agent_executor):
        self.agent_executor = agent_executor
        self.compliance_framework = ComplianceFramework()
        self.audit_logger = ComprehensiveAuditLogger()
        self.security_monitor = SecurityMonitor()
        self.ethics_monitor = EthicsMonitor()
        self.governance_log = []
    
    def invoke(self, user_id, user_input, user_demographic=None):
        """
        Invoca el agente con governance completo.
        
        Args:
            user_id: ID del usuario
            user_input: Input del usuario
            user_demographic: Demográfico del usuario (opcional)
        
        Returns:
            dict con respuesta y metadatos de governance
        """
        governance_start = time.time()
        
        # 1. Verificar compliance
        gdpr_status = self.compliance_framework.check_compliance("GDPR")
        
        if gdpr_status["status"] != "compliant":
            self.audit_logger.log_compliance_event(
                "compliance_check_failed",
                {"regulation": "GDPR", "status": gdpr_status["status"]},
                user_id,
                "critical"
            )
            return {
                "success": False,
                "governance_status": "compliance_failed",
                "message": "❌ Request bloqueada: compliance GDPR no cumplido",
                "response": None
            }
        
        # 2. Detectar anomalías de seguridad
        request_pattern = {
            "frequency": 1,
            "complexity": len(user_input)
        }
        anomaly_check = self.security_monitor.detect_anomalies(user_id, request_pattern)
        
        if anomaly_check["has_anomalies"] and anomaly_check["severity"] == "high":
            self.audit_logger.log_security_event(
                "anomaly_detected",
                {"anomalies": anomaly_check["anomalies"]},
                user_id,
                "high"
            )
            return {
                "success": False,
                "governance_status": "security_anomaly",
                "message": "❌ Request bloqueada: se detectaron anomalías de seguridad",
                "response": None
            }
        
        # 3. Invocar agente
        try:
            response = self.agent_executor.invoke({"input": user_input})
            raw_output = response['output']
            
            # 4. Registrar decisión ética
            self.ethics_monitor.track_decision("query_response", "allowed", user_demographic)
            
            # 5. Actualizar baseline de comportamiento
            self.security_monitor.update_baseline(user_id, request_pattern)
            
            # 6. Registrar evento de governance
            governance_time = time.time() - governance_start
            self._log_governance_event(user_id, user_input, "success", {
                "governance_time": governance_time,
                "compliance_checks": ["GDPR"],
                "security_checks": ["anomaly_detection"]
            })
            
            # 7. Audit logging
            self.audit_logger.log_event(
                "user_activity",
                "agent_invocation",
                {"input_length": len(user_input), "output_length": len(raw_output)},
                user_id,
                "info"
            )
            
            return {
                "success": True,
                "governance_status": "compliant",
                "message": "✅ Request completada con governance completo",
                "response": raw_output,
                "governance_metadata": {
                    "compliance_status": gdpr_status["status"],
                    "security_status": "no_anomalies",
                    "governance_time": governance_time
                }
            }
            
        except Exception as e:
            self.audit_logger.log_security_event(
                "agent_error",
                {"error": str(e)},
                user_id,
                "high"
            )
            return {
                "success": False,
                "governance_status": "error",
                "message": f"❌ Error en el agente: {e}",
                "response": None
            }
    
    def _log_governance_event(self, user_id, input_text, status, details):
        """
        Registra eventos de governance.
        """
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "user_id": user_id,
            "input": input_text[:100],
            "status": status,
            "details": details
        }
        self.governance_log.append(log_entry)
    
    def get_governance_report(self):
        """
        Genera un reporte comprehensivo de governance.
        
        Returns:
            dict con reporte
        """
        return {
            "compliance": {
                "GDPR": self.compliance_framework.check_compliance("GDPR"),
                "SOC_2": self.compliance_framework.check_compliance("SOC_2")
            },
            "audit_stats": self.audit_logger.get_statistics(),
            "security_metrics": self.security_monitor.get_security_metrics(),
            "ethics_metrics": self.ethics_monitor.get_ethics_metrics(),
            "active_alerts": len(self.security_monitor.get_active_alerts()),
            "total_governance_events": len(self.governance_log)
        }

# Crear el agente con governance completo
governed_agent = GovernedAgentWrapper(agent_executor)

print("✅ Agente con Governance Completo configurado.")

## 6. Práctica con el Agente Gobernado

Probamos el agente con governance completo:

In [ ]:
# Pruebas del agente gobernado
governance_test_scenarios = [
    {
        "user_id": "user_gov_1",
        "input": "¿Qué es la inteligencia artificial?",
        "demographic": "general",
        "description": "Consulta normal"
    },
    {
        "user_id": "user_gov_2",
        "input": "¿Cuáles son los principios de la seguridad en IA?",
        "demographic": "technical",
        "description": "Consulta técnica"
    },
    {
        "user_id": "user_gov_3",
        "input": "¿Cómo se implementa compliance GDPR en sistemas IA?",
        "demographic": "compliance",
        "description": "Consulta sobre compliance"
    }
]

print("🧪 Pruebas del Agente con Governance Completo:")
for scenario in governance_test_scenarios:
    print(f"\n--- {scenario['description']} ---")
    print(f"User ID: {scenario['user_id']}")
    print(f"Input: {scenario['input']}")
    print(f"Demographic: {scenario['demographic']}")
    
    result = governed_agent.invoke(
        scenario['user_id'], 
        scenario['input'],
        scenario['demographic']
    )
    print(f"\nEstado de governance: {result['governance_status']}")
    print(f"Mensaje: {result['message']}")
    if result.get('governance_metadata'):
        print(f"Metadata: {result['governance_metadata']}")
    if result['response']:
        print(f"Response: {result['response'][:200]}..." if len(result['response']) > 200 else f"Response: {result['response']}")

## 7. Reporte Comprehensivo de Governance

In [ ]:
# Generar reporte comprehensivo de governance
governance_report = governed_agent.get_governance_report()

print("📊 Reporte Comprehensivo de Governance:")

print("\n--- Compliance ---")
for reg, status in governance_report["compliance"].items():
    print(f"  {reg}: {status['status']}")

print("\n--- Audit Statistics ---")
for key, value in governance_report["audit_stats"].items():
    print(f"  {key}: {value}")

print("\n--- Security Metrics ---")
for key, value in governance_report["security_metrics"].items():
    print(f"  {key}: {value}")

print("\n--- Ethics Metrics ---")
for key, value in governance_report["ethics_metrics"].items():
    print(f"  {key}: {value}")

print(f"\n--- Active Alerts ---")
print(f"  Total: {governance_report['active_alerts']}")

print(f"\n--- Total Governance Events ---")
print(f"  Total: {governance_report['total_governance_events']}")

## 8. Resumen

### Componentes Implementados
- **ComplianceFramework**: Sistema de compliance con regulaciones internacionales (GDPR, CCPA, EU AI Act, SOC 2)
- **ComprehensiveAuditLogger**: Sistema de audit logging multi-categoría con filtrado avanzado
- **SecurityMonitor**: Sistema de monitoring de seguridad con detección de anomalías
- **EthicsMonitor**: Sistema de monitoring ético con análisis de decisiones
- **GovernedAgentWrapper**: Integración completa de governance en el agente

### Regulaciones Soportadas
- **GDPR**: General Data Protection Regulation (UE)
- **CCPA**: California Consumer Privacy Act (USA)
- **EU AI Act**: European Union AI Act (UE)
- **SOC 2**: Service Organization Control 2 (Global)

### Capacidades de Monitoring
- **Security Monitoring**: Detección de anomalías, alertas en tiempo real, métricas de seguridad
- **Ethics Monitoring**: Seguimiento de decisiones éticas, métricas de cumplimiento
- **Audit Logging**: Registro comprehensivo de eventos, filtrado por categoría/severidad/usuario
- **Compliance Tracking**: Verificación de cumplimiento regulatorio, audit trails

### Valor Organizacional
- **Regulatory Compliance**: Cumplimiento con regulaciones internacionales
- **Risk Management**: Detección proactiva de riesgos de seguridad y ética
- **Audit Readiness**: Preparación para auditorías con trails completos
- **Transparency**: Visibilidad completa del comportamiento del sistema
- **Accountability**: Trazabilidad de todas las decisiones y acciones

### Preparación para IL3.4
- **Foundation establecida**: Governance y monitoring como base para escalabilidad
- **Compliance implementado**: Frameworks regulatorios operativos
- **Monitoring activo**: Sistemas de vigilancia en tiempo real
- **Audit trails completos**: Trazabilidad para escalabilidad sostenible